In [ ]:
import os
import csv
import itertools
from dotenv import load_dotenv
import pyarrow as pa
import pyarrow.parquet as pq
from kgxval.dir.Ingest import makeIngestObjsDict
from kgxval.dir.KGXSummarizer import KGXSummarizer

In [ ]:
load_dotenv()
ingest_dict = makeIngestObjsDict(os.environ["INGEST_TOP_LEVEL_DIR"])
hp_cats:tuple[str,...] = tuple([
        "gene or gene product",
        "disease or phenotypic feature",
        "chemical entity",
])

In [ ]:
properties_to_gather_tsv =\
"""Edge Property	KGX Infores Sources	Data Type
adjusted_p_value	ctd	continuous
dgidb_evidence_score	dgidb	continuous
dgidb_interaction_score	dgidb	continuous
diseases_confidence_score	diseases	continuous
evidence_count	tmkp	continuous
has_confidence_score	chembl, cohd, signor	continuous
number_of_cases	dakp	continuous
p_value	ctd	continuous
publications	alliance, bindingdb, chembl, ctd, dakp, dgidb, drugcentral, gene2phenotype, go_cam, goa, gtopdb, hpoa, intact, pubtator, semmeddb, signor, tmkp	continuous (but you ahve to derive a count from the list of pub ids first - as you know)
has_evidence_of_type	goa, hpoa	categorical
gene2phenotype_confidence_category	gene2phenotype	categorical"""

kgx_prop_dicts = list(csv.DictReader(properties_to_gather_tsv.split('\n'),delimiter='\t'))

In [ ]:
def gen_ingest_name_and_prop_name(kgx_prop_dict):
    ingest_names = kgx_prop_dict["KGX Infores Sources"]
    prop = kgx_prop_dict["Edge Property"]
    for ingest_name in ingest_names.split(","):
        ingest_obj = KGXSummarizer.initWithIngestObj(ingest_dict[ingest_name.strip()]["normalized"],hp_cats)
        yield (ingest_obj,ingest_name.strip(),prop)


In [ ]:
from collections import defaultdict


def gen_prop_tuple_from_ingest(ingest_obj: KGXSummarizer, ingest_name:str, prop:str):
    present_cnt = 0
    absent_cnt = 0
    present_spqo_cnts = defaultdict(int)
    absent_spqo_cnts = defaultdict(int)
    for edge in ingest_obj.iter_edges():
        spqo_tup = ingest_obj._makeSPQOFromEdgeDict(edge).makeTuple()
        (sub,pred,qual,obj) = spqo_tup
        if(prop not in edge):
            absent_cnt+=1
            absent_spqo_cnts[spqo_tup]+=1
        if(prop in edge):
            present_cnt+=1
            present_spqo_cnts[spqo_tup]+=1
            present = True
            if(prop=="publications"):
                edge[prop] = len(edge["publications"])
            prop_type = type(edge[prop])
            if(prop_type==type(None)):
                none_val=True
                zero_len=True
                yield(ingest_name, sub, pred, qual, obj, edge[prop], zero_len, none_val)
            elif(prop_type==str):
                none_val=False
                zero_len=(len(edge[prop])==0)
                yield(ingest_name, sub, pred, qual, obj, edge[prop], zero_len, none_val)
            elif((prop_type==int) or (prop_type==float)):
                none_val=False
                zero_len=False
                yield(ingest_name, sub, pred, qual, obj, edge[prop], zero_len, none_val)
            elif((prop_type==list)):
                none_val = False
                zero_len=(len(edge[prop])==0)
                yield(ingest_name, sub, pred, qual, obj, str(edge[prop]), zero_len, none_val)
    with open("data/prop_seen_absent_cnt.csv",'a') as f:
        f.write(f"{ingest_name},{prop},{present_cnt},{absent_cnt}\n")
    with open(f"data/seen_absent_indiv/{ingest_name}-{prop}_cnts.csv", 'w') as f:
        all_spqos = set(present_spqo_cnts.keys()).union(absent_spqo_cnts.keys())
        writer = csv.writer(f)
        writer.writerow(["spqo","present_cnt","absent_cnt"])
        for spqo in sorted(all_spqos):
            writer.writerow([spqo,present_spqo_cnts[spqo],absent_spqo_cnts[spqo]])

def makeSchema(ingest_obj,prop):
    pa_prop_type = None
    for edge in ingest_obj.iter_edges():
        if((prop in edge) and (edge[prop]!=None)):
            prop_type = type(edge[prop])
            if(prop=="publications"):pa_prop_type=pa.int32()
            elif(prop_type==str):
                pa_prop_type = pa.string()
            elif(prop_type==int):
                pa_prop_type = pa.int32()
            elif(prop_type==float):
                pa_prop_type = pa.float32()
            elif(prop_type==list):
                pa_prop_type = pa.string()
            else:
                raise ValueError(f"Need to make a mapping for {prop_type}")
            break
    return pa.schema([
        pa.field("kgx",pa.string()),
        pa.field("sub",pa.string()),
        pa.field("pred",pa.string()),
        pa.field("qual",pa.string()),
        pa.field("obj",pa.string()),
        pa.field(prop, pa_prop_type),
        pa.field("zero_len", pa.bool_()),
        pa.field("na_val",pa.bool_())
    ])

def gen_record_batch(ingest_obj:KGXSummarizer, ingest_name:str, prop:str):
    my_schema = makeSchema(ingest_obj,prop)    
    for batch in itertools.batched(gen_prop_tuple_from_ingest(ingest_obj,ingest_name,prop),1024):
        yield pa.RecordBatch.from_arrays([[x[i] for x in batch] for i in range(8)],schema=my_schema)

In [ ]:
def writePqFile(ingest_obj,ingest_name,prop):
    writer = None
    print(f"Starting {ingest_name}-{prop}")
    for batch in gen_record_batch(ingest_obj, ingest_name, prop):
        if(writer==None):
            writer = pq.ParquetWriter(f'data/parquet/{ingest_name}-{prop}.parquet', batch.schema)
        writer.write_batch(batch)
    if(writer!=None):writer.close()


with open("data/prop_seen_absent_cnt.csv",'w') as f:
    f.write(f"ingest_name,prop_name,present_cnt,absent_cnt\n")
                
for d in kgx_prop_dicts:
    for (ingest_obj,ingest_name,prop) in gen_ingest_name_and_prop_name(d):
        writePqFile(ingest_obj,ingest_name,prop)